In [15]:
from pathlib import Path
import re
import pandas as pd

transcript_path = Path("/Users/nourahmed/Arabic-NLP-System/Transcripts/الساموراي  الدحيح.txt")
qa_path = Path("/Users/nourahmed/Arabic-NLP-System/QA/3AwL93uolIA_QA.csv")

# Read raw transcript lines
with open(transcript_path, "r", encoding="utf-8") as f:
    transcript_lines_raw = f.readlines()

print("Transcript lines:", len(transcript_lines_raw))
print("First 5 raw lines:")
for l in transcript_lines_raw[:5]:
    print(l.rstrip())

# Read QA
qa = pd.read_csv(qa_path)
print("\nQA shape:", qa.shape)
print("QA columns:", qa.columns.tolist())
qa.head()

Transcript lines: 954
First 5 raw lines:
0.167: منذ زمنٍ بعيد،
2.6: في أرضٍ ليست ببعيدة...
6.818: كان هُناك طفلٌ مصريٌ سمين،
10.204: يحلم بأن يكون أول ساموراي مصري في العالم.
17.745: كَبُر الطفل،

QA shape: (300, 6)
QA columns: ['video_id', 'video_title', 'question_id', 'question', 'answer', 'difficulty']


,video_id,video_title,question_id,question,answer,difficulty
0,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q001,ماذا ورد في النص حول هذه الجزئية؟,والساموزين والسامو عليكم!,Easy
1,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q002,ما الجملة المذكورة في هذا الموضع؟,من كل مَن حرمه من حلم الساموراي...,Medium
2,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q003,كيف صيغت العبارة في النص؟,السلام عليكم ورحمة الله وبركاته،,Easy
3,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q004,ما الذي قيل في هذا السياق؟,"من برنامج ""الدحّيح""!",Medium
4,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q005,ما النص الحرفي المذكور هنا؟,خلّيني آخدك مش لمكان واحد،,Easy


In [16]:
def strip_timestamp(line: str) -> str:
    parts = line.split(":", 1)
    if len(parts) == 2:
        return parts[1].strip()
    return line.strip()

transcript_no_ts_lines = [strip_timestamp(l) for l in transcript_lines_raw]
# drop empty lines
transcript_no_ts_lines = [l for l in transcript_no_ts_lines if l]

print("First 5 lines after timestamp removal:")
for l in transcript_no_ts_lines[:5]:
    print(l)

First 5 lines after timestamp removal:
منذ زمنٍ بعيد،
في أرضٍ ليست ببعيدة...
كان هُناك طفلٌ مصريٌ سمين،
يحلم بأن يكون أول ساموراي مصري في العالم.
كَبُر الطفل،


In [17]:

from collections import Counter

# 1. NOISE DETECTION: Counting Stutters and Hesitations
def detect_noise(lines):
    # Pattern for stuttering (e.g., "الـ... الساموراي") or repeated punctuation
    stutter_pattern = r'\b(\w+)\.\.\.' 
    hesitations = [re.findall(stutter_pattern, line) for line in lines]
    flat_hesitations = [item for sublist in hesitations if sublist for item in sublist]
    
    # Pattern for excessive punctuation (!!! or ؟؟؟)
    excl_pattern = r'!{2,}|؟{2,}'
    excessive_punct = [re.findall(excl_pattern, line) for line in lines]
    flat_punct = [item for sublist in excessive_punct if sublist for item in sublist]
    
    print(f"--- Noise Analysis ---")
    print(f"Total Stutters Found: {len(flat_hesitations)}")
    print(f"Top 3 Stuttered Words: {Counter(flat_hesitations).most_common(3)}")
    print(f"Instances of Repeated Punctuation: {len(flat_punct)}\n")

# 2. LINGUISTIC DISTRIBUTION: MSA vs. Egyptian Dialect Indicators
def analyze_dialect_markers(lines):
    # Common Egyptian markers (e.g., 'مش', 'ده', 'اللي', 'يا باشا')
    eg_markers = ['مش', 'ده', 'دي', 'اللي', 'عشان', 'برضه', 'كده']
    # Common MSA markers (e.g., 'ليس', 'هذا', 'الذي', 'لكي')
    msa_markers = ['ليس', 'هذا', 'هذه', 'الذي', 'التي', 'سوف', 'لن']
    
    text = " ".join(lines)
    words = text.split()
    
    eg_count = sum(1 for w in words if w in eg_markers)
    msa_count = sum(1 for w in words if w in msa_markers)
    
    print(f"--- Dialect Distribution (Sample Markers) ---")
    print(f"Total Words: {len(words)}")
    print(f"Egyptian Marker Count: {eg_count}")
    print(f"MSA Marker Count: {msa_count}")
    print(f"Ratio (EG/MSA): {eg_count/msa_count:.2f}" if msa_count > 0 else "N/A")

# RUNNING THE ANALYSIS
# Use the 'transcript_no_ts_lines' from your previous step
detect_noise(transcript_no_ts_lines)
analyze_dialect_markers(transcript_no_ts_lines)

--- Noise Analysis ---
Total Stutters Found: 11
Top 3 Stuttered Words: [('ربما', 2), ('ببعيدة', 1), ('الطفل', 1)]
Instances of Repeated Punctuation: 0

--- Dialect Distribution (Sample Markers) ---
Total Words: 4627
Egyptian Marker Count: 135
MSA Marker Count: 20
Ratio (EG/MSA): 6.75


In [26]:
from difflib import SequenceMatcher

def find_entity_variations(text_lines, threshold=0.8):
    # Extract unique words (excluding very short ones)
    words = set(re.findall(r'\b\w{4,}\b', " ".join(text_lines)))
    variations = []
    
    word_list = list(words)
    for i in range(len(word_list)):
        for j in range(i + 1, len(word_list)):
            w1, w2 = word_list[i], word_list[j]
            # Check similarity ratio
            ratio = SequenceMatcher(None, w1, w2).ratio()
            if ratio > threshold and ratio < 1.0:
                variations.append((w1, w2, ratio))
                
    return sorted(variations, key=lambda x: x[2], reverse=True)

# Run on your transcript
potential_variants = find_entity_variations(transcript_no_ts_lines)
print("Potential Named Entity Variations:")
for v in potential_variants[:20]:
    print(f"'{v[0]}' vs '{v[1]}' (Similarity: {v[2]:.2f})")

Potential Named Entity Variations:
'الـSeppuku' vs 'بالـSeppuku' (Similarity: 0.95)
'الإمبراطور' vs 'الإمبراطوري' (Similarity: 0.95)
'الإمبراطور' vs 'والإمبراطور' (Similarity: 0.95)
'والساموراي' vs 'الساموراي' (Similarity: 0.95)
'والساموراي' vs 'والساموري' (Similarity: 0.95)
'الساموراي' vs 'فالساموراي' (Similarity: 0.95)
'بيستخدموا' vs 'بيستخدموها' (Similarity: 0.95)
'الحرفيين' vs 'والحرفيين' (Similarity: 0.94)
'وبيعتبروه' vs 'بيعتبروه' (Similarity: 0.94)
'الإقطاعي' vs 'الإقطاعية' (Similarity: 0.94)
'اليابانية' vs 'الياباني' (Similarity: 0.94)
'والمفارقة' vs 'المفارقة' (Similarity: 0.94)
'مقاتلين' vs 'ومقاتلين' (Similarity: 0.93)
'والأسلحة' vs 'الأسلحة' (Similarity: 0.93)
'بالسيفين' vs 'السيفين' (Similarity: 0.93)
'يتمرنوا' vs 'بيتمرنوا' (Similarity: 0.93)
'الحقيقية' vs 'الحقيقي' (Similarity: 0.93)
'يتحولوا' vs 'هيتحولوا' (Similarity: 0.93)
'الهزيمة' vs 'بالهزيمة' (Similarity: 0.93)
'اليابان' vs 'الياباني' (Similarity: 0.93)


In [19]:
AR_PUNCT_MAP = {
    "،": ",",
    "؛": ";",
    "؟": "?",
    "“": '"',
    "”": '"',
    "‘": "'",
    "’": "'",
}

def normalize_punct_and_space(text: str) -> str:
    # map punctuation
    for k, v in AR_PUNCT_MAP.items():
        text = text.replace(k, v)

    # normalize ellipsis variants
    text = re.sub(r"\.{2,}", "...", text)

    # remove weird spacing
    text = re.sub(r"\s+", " ", text).strip()

    return text

transcript_punct_lines = [normalize_punct_and_space(l) for l in transcript_no_ts_lines]
print(transcript_punct_lines[:5])

['منذ زمنٍ بعيد,', 'في أرضٍ ليست ببعيدة...', 'كان هُناك طفلٌ مصريٌ سمين,', 'يحلم بأن يكون أول ساموراي مصري في العالم.', 'كَبُر الطفل,']


In [20]:
# Arabic diacritics (tashkeel) + tatweel
DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")  # includes tatweel

def normalize_arabic(text: str) -> str:
    # remove diacritics/tatweel
    text = re.sub(DIACRITICS_RE, "", text)

    # normalize hamza/alef forms to bare alef
    text = re.sub(r"[إأآٱ]", "ا", text)

    # normalize ya/alef maqsura
    text = text.replace("ى", "ي")

    # normalize ta marbuta (choose ONE policy; this one maps ة→ه)
    text = text.replace("ة", "ه")

    # normalize waw/ya hamza to base letters (optional but helps consistency)
    text = text.replace("ؤ", "و").replace("ئ", "ي")

    return text

transcript_norm_lines = [normalize_arabic(l) for l in transcript_punct_lines]
print(transcript_norm_lines[:5])

['منذ زمن بعيد,', 'في ارض ليست ببعيده...', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,']


In [21]:
def normalize_repeated_chars(text: str) -> str:
    # any char repeated 3+ times -> 2 times (e.g., "جمييييل" -> "جمييل")
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

transcript_norm_lines = [normalize_repeated_chars(l) for l in transcript_norm_lines]
print(transcript_norm_lines[:10])

['منذ زمن بعيد,', 'في ارض ليست ببعيده..', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,', 'وسافر ل"اليابان",', 'رفضه كل معلمي الساموراي', 'والساموزين والسامو عليكم!', '"السلام عليكم"', 'تاجج الغضب بداخل الطفل..']


In [ ]:
# removee extra spaces around latin words to avoid merging them with Arabic
def normalize_english_case(text: str) -> str:
    # lower-case latin sequences only
    text = re.sub(r'([\u0600-\u06FF])([A-Za-z])', r'\1 \2', text)
    text = re.sub(r'([A-Za-z])([\u0600-\u06FF])', r'\1 \2', text)
    def lower_latin(m):
        return m.group(0).lower()
    return re.sub(r"[A-Za-z]+", lower_latin, text)

transcript_norm_lines = [normalize_english_case(l) for l in transcript_norm_lines]
print(transcript_norm_lines[:20])

['منذ زمن بعيد,', 'في ارض ليست ببعيده..', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم.', 'كبر الطفل,', 'وسافر ل"اليابان",', 'رفضه كل معلمي الساموراي', 'والساموزين والسامو عليكم!', '"السلام عليكم"', 'تاجج الغضب بداخل الطفل..', 'اقسم بان ينتقم', 'من كل من حرمه من حلم الساموراي..', 'خلع رداءه,', 'وابتدا بداء جديدا..', 'وحينها, قرر..', 'انه لن ياكل العسل مره اخري..', 'بل ربما..', 'ربما.. يدس فيه شييا.', 'اعزايي المشاهدين,', 'السلام عليكم ورحمه الله وبركاته,']


In [27]:
import re

def clean_stutters_and_noise(text):
    # 1. Remove stutters (e.g., "السامو... الساموراي" -> "السامو الساموراي")
    text = re.sub(r'\b(\w+)\.\.\.\s+\1\b', r'\1', text)
    # very common in transcribed audio where a speaker hesitates.

    # 2. Reduce repeated punctuation (e.g., !!! -> ! or ؟؟؟ -> ؟)
    text = re.sub(r'!{2,}', '!', text)
    text = re.sub(r'؟{2,}', '؟', text)
    
    # 3. Handle Punctuation Spacing (Isolate from words)
    # This ensures "الساموراي!" becomes "الساموراي !"
    text = re.sub(r'([،؟!.()"-])', r' \1 ', text)

# In Machine Learning (NLP), "الساموراي" and "الساموراي!" are seen as two different words. By adding spaces, the computer can treat the word and the punctuation as two separate "tokens."
    
    return text
transcript_cleaned_lines = [clean_stutters_and_noise(l) for l in transcript_norm_lines]
print(transcript_cleaned_lines[:20])


['منذ زمن بعيد,', 'في ارض ليست ببعيده .  . ', 'كان هناك طفل مصري سمين,', 'يحلم بان يكون اول ساموراي مصري في العالم . ', 'كبر الطفل,', 'وسافر ل " اليابان " ,', 'رفضه كل معلمي الساموراي', 'والساموزين والسامو عليكم ! ', ' " السلام عليكم " ', 'تاجج الغضب بداخل الطفل .  . ', 'اقسم بان ينتقم', 'من كل من حرمه من حلم الساموراي .  . ', 'خلع رداءه,', 'وابتدا بداء جديدا .  . ', 'وحينها, قرر .  . ', 'انه لن ياكل العسل مره اخري .  . ', 'بل ربما .  . ', 'ربما .  .  يدس فيه شييا . ', 'اعزايي المشاهدين,', 'السلام عليكم ورحمه الله وبركاته,']
